In [3]:
from  pathlib import Path
import numpy as np 
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import seaborn as sns
from ipywidgets import interact, widgets
from IPython.display import display, clear_output

In [4]:
exp = 'data_embd_non_opt'
data_dir = Path('/data/horse/ws/miha493e-research_project/activediffusion/data')
data_in = data_dir / exp / 'hdf5' / 'molecules-fps-40000-input.hdf5'
data_out = data_dir / exp / 'hdf5' / 'molecules-fps-40000-output.hdf5'

In [5]:
def extract_properties2(file_path, property1, property2):
    """
    Extract arrays for 'HLgap' and 'eMBD' from the specified HDF5 file and dataset.

    Parameters:
    -----------
    file_path : str
        Path to the HDF5 file.
    property1 : str
        The first property to extract (e.g., 'HLgap').
    property2 : str
        The second property to extract (e.g., 'eMBD').

    Returns:
    --------
    tuple
        Two lists containing the extracted values for the specified properties.
    """

    property1_values = []
    property2_values = []

    with h5py.File(file_path, "r") as f:
        for subset in f.keys():
            if isinstance(f[subset], h5py.Group):
                for geom in f[subset].keys():
                    if property1 in f[subset][geom].keys() and property2 in f[subset][geom].keys():
                        prop1 = f[subset][geom][property1][()]
                        prop2 = f[subset][geom][property2][()]

                        # Convert to scalar if numpy array with zero dims
                        if hasattr(prop1, 'shape') and prop1.shape == ():
                            prop1 = float(prop1)
                        if hasattr(prop2, 'shape') and prop2.shape == ():
                            prop2 = float(prop2)

                        property1_values.append(prop1)
                        property2_values.append(prop2)

    return property1_values, property2_values

In [ ]:
# KDE plots of input vs output eMBD feature
def show_kde(prop1_name, prop1_in_values, prop1_out_values, prop2_name, prop2_in_values, prop2_out_values):
    """
    Generate scatter plots comparing two properties from input and output datasets.
    Parameters:
    -----------
    prop1_name : str
        Name of the first property (e.g., 'HLgap').
    prop1_in_values : list
        Values of the first property from the input dataset.
    prop1_out_values : list
        Values of the first property from the output dataset.
    prop2_name : str
        Name of the second property (e.g., 'eMBD').
    prop2_in_values : list
        Values of the second property from the input dataset.
    prop2_out_values : list
        Values of the second property from the output dataset.
    """
    if prop1_name == 'atNUM':
        prop1_in_values = [len(v) for v in prop1_in_values]
        prop1_out_values = [len(v) for v in prop1_out_values]
    if prop2_name == 'atNUM':
        prop2_in_values = [len(v) for v in prop2_in_values]
        prop2_out_values = [len(v) for v in prop2_out_values]

    prop1_in_values = np.array(prop1_in_values).flatten()
    prop1_out_values = np.array(prop1_out_values).flatten()
    prop2_in_values = np.array(prop2_in_values).flatten()
    prop2_out_values = np.array(prop2_out_values).flatten()

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Input data
    sns.kdeplot(x=prop1_in_values, y=prop2_in_values,
                fill=True, cmap="Blues", thresh=0.05, levels=10, ax=axes[0])
    axes[0].set_title(f'Input Data: {prop1_name} vs {prop2_name}')
    axes[0].set_xlabel(prop1_name)
    axes[0].set_ylabel(prop2_name)
    
    # Output data
    sns.kdeplot(x=prop1_out_values, y=prop2_out_values,
                fill=True, cmap="Reds", thresh=0.05, levels=10, ax=axes[1])
    axes[1].set_title(f'Output Data: {prop1_name} vs {prop2_name}')
    axes[1].set_xlabel(prop1_name)
    axes[1].set_ylabel(prop2_name)
    
    # Overlay with contours
    sns.kdeplot(x=prop1_in_values, y=prop2_in_values,
                fill=False, color="blue", thresh=0.05, levels=10, linewidths=2, ax=axes[2], label='Input')
    sns.kdeplot(x=prop1_out_values, y=prop2_out_values,
                fill=False, color="red", thresh=0.05, levels=10, linewidths=2, ax=axes[2], label='Output')
    axes[2].set_title(f'Overlay: {prop1_name} vs {prop2_name}')
    axes[2].set_xlabel(prop1_name)
    axes[2].set_ylabel(prop2_name)
    axes[2].legend()
    
    plt.tight_layout()
    plt.savefig(f"/data/horse/ws/miha493e-research_project/activediffusion/results/KDE2_{prop1_name}_vs_{prop2_name}_{exp}.png", dpi=300)
    plt.show()


In [9]:
# Dropdown for property selection
molecule_prop = [
    'DIP',
    'HLgap',
    'atNUM',
    'atXYZ',
    'eAT',
    'eC',
    'eEE',
    'eH',
    'eKIN',
    'eL',
    'eMBD',
    'eNE',
    'eNN',
    'ePBE0',
    'ePBE0+MBD',
    'eTS',
    'eX',
    'eXC',
    'eXX',
    'mC6',
    'mPOL'
]
prop1_dropdown = widgets.Dropdown(options=molecule_prop, description='Molecule Property 1:', style={'description_width': 'initial'})
prop2_dropdown = widgets.Dropdown(options=molecule_prop, description='Molecule Property 2:', style={'description_width': 'initial'})
show_button = widgets.Button(description="Show Plot")
# Output area for displaying data
output2 = widgets.Output()

# Callback function for molecule selection
def on_prop1_change(change):
    """
    Callback function to handle changes in the properties dropdown.
    """
    global x_axis
    if change['type'] == 'change' and change['name'] == 'value':
        x_axis = change['new']

def on_prop2_change(change):
    """
    Callback function to handle changes in the properties dropdown.
    """
    global y_axis
    if change['type'] == 'change' and change['name'] == 'value':
        y_axis = change['new']
        
def on_show_plot(b):
    """
    Callback function to handle the show plot button click.
    """
    global x_axis, y_axis
    with output2:
        clear_output()
        x_in_values, y_in_values = extract_properties2(data_in, x_axis, y_axis)
        x_out_values, y_out_values = extract_properties2(data_out, x_axis, y_axis)
        show_kde(x_axis, x_in_values, x_out_values, y_axis, y_in_values, y_out_values)

        

# Connect callbacks
prop1_dropdown.observe(on_prop1_change)
prop2_dropdown.observe(on_prop2_change)
show_button.on_click(on_show_plot)


# Display widgets and output
display(widgets.VBox([prop1_dropdown, widgets.HBox((prop2_dropdown, show_button)), output2]))
x_axis = prop1_dropdown.value
y_axis = prop2_dropdown.value